# Final Data Cleaning — leftpolitics_with_descriptions.csv
Loads `Data/non fiction/leftpolitics_with_descriptions.csv`, filters to English/English-translation books, handles missing values, and saves to `Data/non fiction/leftpolitics_final_clean.csv`.

In [1]:
import os
import pandas as pd
from langdetect import detect, LangDetectException

INPUT_PATH  = "../Data/non fiction/leftpolitics_with_descriptions.csv"
OUTPUT_PATH = "../Data/non fiction/leftpolitics_final_clean.csv"

In [2]:
df = pd.read_csv(INPUT_PATH)
print(f"Loaded {len(df)} rows, {df.shape[1]} columns")
print("\nColumns:", df.columns.tolist())
print("\nMissing values:")
print(df.isnull().sum())
print(f"\nDuplicates: {df.duplicated().sum()}")
df.head(3)

Loaded 5487 rows, 8 columns

Columns: ['title', 'author', 'year_published', 'subjects', 'edition_count', 'open_library_key', 'queried_author', 'description']

Missing values:
title                  0
author                 1
year_published      3933
subjects            5487
edition_count          0
open_library_key       0
queried_author         0
description         1314
dtype: int64

Duplicates: 0


,title,author,year_published,subjects,edition_count,open_library_key,queried_author,description
0,La naissance du maquis dans le Sud-Cameroun (H...,Achille Mbembe,1996.0,NaN,2,/book/show/74573817-la-naissance-du-maquis-dan...,Achille Mbembe,Cet ouvrage s'efforce de traquer les formes pu...
1,On the Postcolony,Achille Mbembe,2001.0,NaN,17,/book/show/149757.On_the_Postcolony,Achille Mbembe,Achille Mbembe is one of the most brilliant th...
2,Johannesburg: The Elusive Metropolis (Volume 16),Sarah Nuttall,2004.0,NaN,5,/book/show/327816.Johannesburg,Achille Mbembe,This issue of Public Culture attempts to overt...


## Basic cleaning

In [3]:
df_clean = df.copy()

# Strip whitespace from all string columns
str_cols = df_clean.select_dtypes(include="object").columns
df_clean[str_cols] = df_clean[str_cols].apply(lambda col: col.str.strip())

# Drop subjects — 100 % null, no useful information
df_clean = df_clean.drop(columns=["subjects"])

# Standardize author/queried_author casing and collapse extra spaces
def standardize_name(name):
    return " ".join(str(name).title().split())

df_clean["author"] = df_clean["author"].apply(standardize_name)
df_clean["queried_author"] = df_clean["queried_author"].apply(standardize_name)

# Ensure year is numeric
df_clean["year_published"] = pd.to_numeric(df_clean["year_published"], errors="coerce")

# Drop rows missing title or author
before = len(df_clean)
df_clean = df_clean.dropna(subset=["title", "author"])
df_clean = df_clean[df_clean["title"].str.len() > 0]
df_clean = df_clean[df_clean["author"].str.len() > 0]
print(f"Dropped {before - len(df_clean)} rows missing title or author → {len(df_clean)} remain")

# Drop rows missing description — needed for recommendations
before = len(df_clean)
df_clean = df_clean.dropna(subset=["description"])
df_clean = df_clean[df_clean["description"].str.len() > 0]
print(f"Dropped {before - len(df_clean)} rows missing description → {len(df_clean)} remain")

# Drop exact duplicates (title + author)
before = len(df_clean)
df_clean = df_clean.drop_duplicates(subset=["title", "author"])
print(f"Dropped {before - len(df_clean)} duplicate rows → {len(df_clean)} remain")

Dropped 0 rows missing title or author → 5487 remain
Dropped 1314 rows missing description → 4173 remain
Dropped 2 duplicate rows → 4171 remain


/var/folders/b7/yg62l3k53kvc1lvmnwswr23m0000gn/T/ipykernel_76710/1400259018.py:4: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  str_cols = df_clean.select_dtypes(include="object").columns


## Language filtering — keep English and English translations

Detect language from the description (most reliable signal). Books in English or translated into English are kept.

In [4]:
def detect_lang(text):
    try:
        return detect(str(text))
    except LangDetectException:
        return "unknown"

df_clean["lang"] = df_clean["description"].apply(detect_lang)

print("Language distribution (top 10):")
print(df_clean["lang"].value_counts().head(10))

before = len(df_clean)
df_clean = df_clean[df_clean["lang"] == "en"].drop(columns=["lang"])
print(f"\nDropped {before - len(df_clean)} non-English rows → {len(df_clean)} remain")

Language distribution (top 10):
lang
en    2960
es     258
fr     223
pt     152
de     127
it     106
ar      76
nl      39
tr      36
af      22
Name: count, dtype: int64

Dropped 1211 non-English rows → 2960 remain


## Final check and save

In [5]:
df_clean = df_clean.reset_index(drop=True)

print("=== Final dataset ===")
print(f"Shape: {df_clean.shape}")
print("\nMissing values:")
print(df_clean.isnull().sum())
print(f"\nDuplicates: {df_clean.duplicated().sum()}")
df_clean.head()

=== Final dataset ===
Shape: (2960, 7)

Missing values:
title                  0
author                 0
year_published      1848
edition_count          0
open_library_key       0
queried_author         0
description            0
dtype: int64

Duplicates: 0


,title,author,year_published,edition_count,open_library_key,queried_author,description
0,On the Postcolony,Achille Mbembe,2001.0,17,/book/show/149757.On_the_Postcolony,Achille Mbembe,Achille Mbembe is one of the most brilliant th...
1,Johannesburg: The Elusive Metropolis (Volume 16),Sarah Nuttall,2004.0,5,/book/show/327816.Johannesburg,Achille Mbembe,This issue of Public Culture attempts to overt...
2,The Earthly Community: Reflections on the Last...,Achille Mbembe,2023.0,15,/book/show/211097016-the-earthly-community,Achille Mbembe,In The Earthly Community: Reflections on the L...
3,Critique of Black Reason,Achille Mbembe,NaN,25,/book/show/30757980-critique-of-black-reason,Achille Mbembe,In Critique of Black Reason eminent critic Ach...
4,Out of the Dark Night: Essays on Decolonization,Achille Mbembe,NaN,17,/book/show/44055182-out-of-the-dark-night,Achille Mbembe,Originally published in French as: Sortir de l...


In [6]:
os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)
df_clean.to_csv(OUTPUT_PATH, index=False)
print(f"Saved {len(df_clean)} rows to {OUTPUT_PATH}")

Saved 2960 rows to ../Data/non fiction/leftpolitics_final_clean.csv
